# Learning-Based Demand Response Bidding Experiment

This notebook runs a demand-forecast-driven bidding experiment for a residential aggregator in a local flexibility market. It loads customer-level demand forecast series, constructs flexibility quantities from forecasted baselines, trains the online-learning components on the training period, evaluates on the test period, and exports day-level, pair-level, and customer-level results.


## How to run

Place the dataset CSV in the `data/` folder or update `CSV_IN` below. The consumer ID columns are treated as demand forecast series produced by upstream forecasting models.


In [ ]:
import math, random, os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def resolve_data_path(csv_in):
    candidate_paths = [
        csv_in,
        os.path.join('data', csv_in),
        os.path.join('/mnt/data', csv_in),
        os.path.join('/content', csv_in),
    ]
    for p in candidate_paths:
        if os.path.exists(p):
            return p
    raise FileNotFoundError(f'Could not find dataset CSV: {csv_in}')

# ============================================================
# Helper: build day plan (96 slots) from pairs_df + original df
# ============================================================
def build_day_plan(df_full, pairs_df, target_date_str):
    target_date = pd.to_datetime(target_date_str).date()

    day_df = df_full.copy()
    day_df["date"] = day_df["datetime"].dt.date
    day_df = day_df[day_df["date"] == target_date].copy().sort_values("datetime")
    if len(day_df) != 96:
        raise ValueError(
            f"Target day {target_date_str} not available as complete day (need 96 rows). Found {len(day_df)}."
        )

    day_df["slot"] = day_df.groupby("date").cumcount().astype(int)

    plan = day_df[["datetime", "slot", "mcp_price", "temp_c"]].copy()
    plan["bids_steps"] = ""
    plan["bid_Q_total"] = 0.0
    plan["incentive_avg_pred"] = 0.0
    plan["offers_sent"] = 0
    plan["pct_zero_incentive"] = 0.0
    plan["target_clear"] = 0.0
    plan["t_low"] = -1

    pday = pairs_df[pairs_df["date"] == str(target_date)].copy()
    if len(pday) == 0:
        raise ValueError(f"No rows in pairs_df for {target_date_str}.")

    for _, r in pday.iterrows():
        tp = int(r["t_peak"])
        tl = int(r["t_low"])
        steps = str(r.get("bid_steps","") or "")

        plan.loc[plan["slot"] == tp, "bids_steps"] = steps
        plan.loc[plan["slot"] == tp, "t_low"] = tl
        plan.loc[plan["slot"] == tp, "incentive_avg_pred"] = float(r.get("incentive_avg_pred", 0.0))
        plan.loc[plan["slot"] == tp, "offers_sent"] = int(r.get("offers_sent", 0))
        plan.loc[plan["slot"] == tp, "pct_zero_incentive"] = float(r.get("pct_zero_incentive", 0.0))
        plan.loc[plan["slot"] == tp, "target_clear"] = float(r.get("target_clear", 0.0))

        if steps.strip():
            qtot = 0.0
            for item in steps.split(";"):
                q_str, _ = item.split("@")
                qtot += float(q_str)
            plan.loc[plan["slot"] == tp, "bid_Q_total"] = qtot

    return plan


# ============================================================
# MAIN: run the whole system on ONE group CSV
# ============================================================
def run_system(csv_in, out_prefix, target_date_str="2025-09-30",
               SPLIT_YEAR=2025, MAX_DAYS=None, seed=13):
    """
    Runs training+test simulation on a given group CSV and exports:
      - {out_prefix}_dr_day_metrics_train_test.csv
      - {out_prefix}_dr_pair_log_train_test.csv
      - {out_prefix}_dr_customer_daily_train_test.csv
      - {out_prefix}_day_plan_{target_date}.csv   (96 rows: slots)
    """

    # -------------------------
    # 0) Seeds / RNG reset
    # -------------------------
    rng = np.random.default_rng(seed)
    random.seed(seed)

    # -------------------------
    # 1) Load + filter cols
    # -------------------------
    df = pd.read_csv(resolve_data_path(csv_in))

    for c in ["datetime", "mcp_price", "temp_c"]:
        if c not in df.columns:
            raise ValueError(f"Missing required column: {c}")

    df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
    df = df.dropna(subset=["datetime"]).sort_values("datetime").reset_index(drop=True)

    cols = list(df.columns)
    i_dt   = cols.index("datetime")
    i_temp = cols.index("temp_c")
    if i_temp <= i_dt + 1:
        raise ValueError("No columns found between datetime and temp_c")

    # demand forecast columns are columns between datetime and temp_c
    demand_forecast_cols = cols[i_dt + 1 : i_temp]

    # IMPORTANT: keep ALL demand forecast series present in this group file
    KEEP_N_CUSTOMERS = len(demand_forecast_cols)
    cust_cols = demand_forecast_cols[:KEEP_N_CUSTOMERS]
    print("Demand forecast series kept (ALL in file):", len(cust_cols))

    # numeric coercion
    for c in cust_cols + ["mcp_price", "temp_c"]:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    print("Loaded rows:", len(df))
    print("Demand forecast series kept:", len(cust_cols))

    # -------------------------
    # 2) Keep only complete days (96 MTUs/day)
    # -------------------------
    df["date"] = df["datetime"].dt.date
    counts = df.groupby("date").size()
    good_days = counts[counts == 96].index
    df = df[df["date"].isin(good_days)].sort_values("datetime").reset_index(drop=True)

    all_days = list(sorted(df["date"].unique()))
    print(f"Complete days available: {len(all_days)}")

    # Optional cap (runtime control)
    if MAX_DAYS is not None and len(all_days) > MAX_DAYS:
        all_days = all_days[:MAX_DAYS]
        df = df[df["date"].isin(all_days)].copy().sort_values("datetime").reset_index(drop=True)
        print(f"Capped to first {MAX_DAYS} days. Using: {len(all_days)}")

    # slot 0..95 within day
    df["slot"] = (df.groupby("date").cumcount()).astype(int)

    # -------------------------
    # 2.5) Calendar train/test split
    # Train: Jan–Aug, Test: Sep of SPLIT_YEAR
    # -------------------------
    day_ts = pd.to_datetime(pd.Series(all_days).astype(str))
    day_year  = day_ts.dt.year.to_numpy()
    day_month = day_ts.dt.month.to_numpy()

    train_mask = (day_year == SPLIT_YEAR) & (day_month >= 1) & (day_month <= 8)
    test_mask  = (day_year == SPLIT_YEAR) & (day_month == 9)

    train_days = [d for d, m in zip(all_days, train_mask) if m]
    test_days  = [d for d, m in zip(all_days, test_mask) if m]

    print("Split year:", SPLIT_YEAR)
    print("Train days:", len(train_days), "Test days:", len(test_days))
    if len(train_days):
        print("Train range:", train_days[0], "->", train_days[-1])
    if len(test_days):
        print("Test  range:", test_days[0], "->", test_days[-1])

    if len(train_days) == 0 or len(test_days) == 0:
        raise ValueError(
            "Train/Test split produced empty set. "
            "Fix by: (a) removing MAX_DAYS cap, or (b) checking that year 2025 Jan–Sep exists as complete days."
        )

    # -------------------------
    # 3) Customer flexibility parameters from demand forecast baselines
    # -------------------------
    price_col = "mcp_price"
    temp_col  = "temp_c"

    param_days = all_days
    df_param = df[df["date"].isin(param_days)].copy().sort_values("datetime").reset_index(drop=True)

    param_all_days = list(sorted(df_param["date"].unique()))
    n_days = len(param_all_days)
    n_cust = len(cust_cols)

    demand_forecast_matrix = df_param[cust_cols].to_numpy(dtype=float)  # (n_days*96, n_cust)
    demand_forecast_matrix = demand_forecast_matrix.reshape(n_days, 96, n_cust)  # (days, slots, cust)

    demand_forecast_matrix = np.where(np.isfinite(demand_forecast_matrix), demand_forecast_matrix, np.nan)
    demand_forecast_matrix = np.where(demand_forecast_matrix < 0, 0.0, demand_forecast_matrix)

    EVE_START, EVE_END = 17*4, 23*4
    eve_idx = np.arange(EVE_START, EVE_END)

    daily_total = np.nansum(demand_forecast_matrix, axis=1)                 # (days, cust)
    eve_total   = np.nansum(demand_forecast_matrix[:, eve_idx, :], axis=1)  # (days, cust)

    dows = np.array([pd.Timestamp(d).weekday() for d in param_all_days])
    is_weekend = np.isin(dows, [5, 6])

    def iqr(a, axis=0):
        q75 = np.nanquantile(a, 0.75, axis=axis)
        q25 = np.nanquantile(a, 0.25, axis=axis)
        return q75 - q25

    eve_slot_iqr = iqr(demand_forecast_matrix[:, eve_idx, :], axis=0)
    eve_slot_med = np.nanmedian(demand_forecast_matrix[:, eve_idx, :], axis=0)

    var_proxy = np.nanmedian(eve_slot_iqr / (eve_slot_med + 1e-6), axis=0)
    var_proxy = np.clip(var_proxy, 0.0, 3.0)
    var01 = var_proxy / (var_proxy + 1.0)

    eve_share = np.nanmedian(eve_total / (daily_total + 1e-9), axis=0)
    eve_share = np.clip(eve_share, 0.0, 1.0)

    wk_eve = np.nanmedian(eve_total[is_weekend, :], axis=0)
    wd_eve = np.nanmedian(eve_total[~is_weekend, :], axis=0)
    weekend_mult = (wk_eve + 1e-6) / (wd_eve + 1e-6)
    weekend_mult = np.clip(weekend_mult, 1.0, 1.35)

    base_daily = np.nanmedian(daily_total, axis=0)
    base_daily = np.where(np.isfinite(base_daily), base_daily, np.nanmedian(base_daily[np.isfinite(base_daily)]))

    FLEX_SCALE = 0.25
    flex_day = FLEX_SCALE * base_daily * (0.6 + 1.4*eve_share) * (0.4 + 1.2*var01)
    flex_day = np.clip(flex_day, 0.15, 0.20*base_daily)

    sensitivity = 0.6 + 2.0*(0.55*var01 + 0.45*eve_share)
    sensitivity = np.clip(sensitivity, 0.5, 3.0)

    curvature = 0.9 - 0.35*var01
    curvature = np.clip(curvature, 0.45, 1.0)

    eve_bias = 0.05 + 0.30*np.clip((eve_share - 0.15)/0.45, 0.0, 1.0)
    eve_bias = np.clip(eve_bias, 0.05, 0.35)

    eve_slot_std = np.nanmedian(np.nanstd(demand_forecast_matrix[:, eve_idx, :], axis=0), axis=0)
    noise_rel = np.clip((eve_slot_std / (np.nanmedian(demand_forecast_matrix[:, eve_idx, :], axis=(0,1)) + 1e-6)), 0.10, 0.60)

    print("Param preview:")
    print("  flex_day median:", float(np.nanmedian(flex_day)))
    print("  sensitivity median:", float(np.nanmedian(sensitivity)))
    print("  curvature median:", float(np.nanmedian(curvature)))
    print("  weekend_mult median:", float(np.nanmedian(weekend_mult)))
    print("  customers:", n_cust)

    # -------------------------
    # 4) CONFIG
    # -------------------------
    N_PEAK, N_LOW = 16, 16
    INCENTIVES = [0.00, 0.03, 0.06, 0.09, 0.12, 0.15, 0.18]
    AGG_TARGETS = [0.45, 0.55, 0.65]

    MAX_STEPS_PER_DAY = 6
    FALLBACK_Q_FRAC = 0.50
    FALLBACK_BID_Q = 0.50

    ALPHA_CVAR = 0.20
    STEPS_PER_CURVE = 5
    MC_SAMPLES = 400

    MAX_DOWNSHIFT_FRAC_PER_MTU = 0.30
    MIN_LOAD_MTU = 0.0

    EPS0, EPS_MIN, EPS_TAU = 0.15, 0.02, 60.0
    def eps_schedule(day_index):
        return max(EPS_MIN, EPS0*math.exp(-day_index/EPS_TAU))

    LOW_SPREAD_STOP = 0.04
    PENALTY_UNDER = 0.02
    MAX_OFFERS_PER_DAY_PER_CUSTOMER = 8

    WARMUP_DAYS = 21
    WARMUP_P_FORCE_NONZERO = 0.60
    WARMUP_MIN_NONZERO_ARM = 1  # 0.03 €/kWh

    # -------------------------
    # 5) Customer class
    # -------------------------
    class DataDrivenCustomer:
        def __init__(self, flex_day, sensitivity, curvature, eve_bias, weekend_mult, noise_rel):
            self.flex_day = float(flex_day)
            self.sensitivity = float(sensitivity)
            self.curvature = float(curvature)
            self.eve_bias = float(eve_bias)
            self.weekend_mult = float(weekend_mult)
            self.noise_rel = float(noise_rel)

        def ctx_scale(self, hour_peak, dow):
            s = math.sin(2*math.pi*hour_peak/24.0)
            scale = 1.0 + max(0.0, self.eve_bias*s)
            if dow in (5, 6):
                scale *= self.weekend_mult
            return scale

        def exp_shift(self, a, hour_peak, dow):
            if a <= 1e-12:
                return 0.0
            base = self.flex_day * self.ctx_scale(hour_peak, dow)
            resp = base * (1 - math.exp(- self.sensitivity * a)) ** self.curvature
            return float(min(resp, 0.35 * self.flex_day))

        def realized(self, a, hour_peak, dow):
            if a <= 1e-12:
                return 0.0
            mu = self.exp_shift(a, hour_peak, dow)
            sigma = max(0.05, self.noise_rel) * max(0.05, mu)
            eps = max(-0.8*mu, rng.normal(0, sigma))
            return max(0.0, mu + eps)

    customers = [
        DataDrivenCustomer(flex_day[i], sensitivity[i], curvature[i], eve_bias[i], weekend_mult[i], noise_rel[i])
        for i in range(n_cust)
    ]

    # -------------------------
    # 6) Forecast generator
    # -------------------------
    def build_forecast_quantiles(past_actual_prices, base_sigma=0.006, K=7):
        if len(past_actual_prices) == 0:
            mu = np.ones(96) * df[price_col].mean()
            sigma = np.ones(96) * 0.01
        else:
            last_K = past_actual_prices[-min(K, len(past_actual_prices)):]
            stack = np.vstack(last_K)
            mu = np.nanmean(stack, axis=0)
            sigma = np.nanstd(stack, axis=0)
            sigma = np.maximum(sigma, base_sigma)

        qdict = {}
        zgrid = np.random.normal(0, 1, 20000)
        for q in [0.1, 0.2, 0.35, 0.5, 0.65, 0.8, 0.9]:
            z = float(np.quantile(zgrid, q))
            qdict[q] = mu + z*sigma
        return mu, qdict, sigma

    # -------------------------
    # 7) LinTS + OnlineVar + Utils
    # -------------------------
    class LinTS:
        def __init__(self, actions, d, v=0.4, l2=1.0):
            self.actions = list(actions)
            self.K = len(self.actions)
            self.d = d
            self.v = v
            self.A = [np.eye(d)*l2 for _ in range(self.K)]
            self.b = [np.zeros((d,)) for _ in range(self.K)]

        def update(self, k, x, r):
            x = x.reshape(-1,)
            self.A[k] += np.outer(x, x)
            self.b[k] += r*x

    class OnlineVar:
        def __init__(self):
            self.n = 0
            self.mean = 0.0
            self.M2 = 0.0
        def update(self, x):
            self.n += 1
            dx = x - self.mean
            self.mean += dx / self.n
            self.M2 += dx * (x - self.mean)
        def var(self):
            return (self.M2 / (self.n - 1)) if self.n > 1 else 0.0
        def std(self, floor=0.005):
            return max(floor, math.sqrt(self.var()))

    def cvar_of_profits(samples, alpha=ALPHA_CVAR):
        s = np.sort(samples)
        idx = max(1, int(math.ceil(alpha*len(s))))
        tail = s[:idx]
        return float(tail.mean()) if len(tail) > 0 else 0.0

    def cyc_feat(x, period):
        return (math.sin(2*math.pi*x/period), math.cos(2*math.pi*x/period))

    D_FEAT = 8

    def gini(x):
        x = np.asarray(x, dtype=float)
        if np.allclose(x.sum(), 0.0):
            return 0.0
        x = np.sort(x)
        n = x.size
        cumx = np.cumsum(x)
        return float((n + 1 - 2*np.sum(cumx)/cumx[-1]) / n)

    # MTU cap helper
    def mtu_cap_kwh(day_idx_in_paramY, tp, ci):
        baseline_forecast = float(demand_forecast_matrix[day_idx_in_paramY, tp, ci])
        if (not np.isfinite(baseline_forecast)) or (baseline_forecast < MIN_LOAD_MTU):
            return 0.0
        return MAX_DOWNSHIFT_FRAC_PER_MTU * baseline_forecast

    # -------------------------
    # 7.5) Bandits init
    # -------------------------
    cust_bandits = [LinTS(INCENTIVES, d=D_FEAT, v=0.4, l2=1.0) for _ in range(n_cust)]
    agg_bandit  = LinTS(AGG_TARGETS, d=D_FEAT, v=0.5, l2=1.0)
    cust_shift_noise = [[OnlineVar() for _ in range(len(INCENTIVES))] for _ in range(n_cust)]

    def expected_shift_from_bandit(ci, a_idx, x):
        Ainv = np.linalg.inv(cust_bandits[ci].A[a_idx])
        mu   = Ainv @ cust_bandits[ci].b[a_idx]
        m    = max(0.0, float(mu @ x))
        return min(m, customers[ci].flex_day)

    def ts_sample_shift(bandit, a_idx, x):
        Ainv = np.linalg.inv(bandit.A[a_idx])
        mu   = Ainv @ bandit.b[a_idx]
        theta = rng.multivariate_normal(mu, (bandit.v**2)*Ainv)
        return max(0.0, float(theta @ x))

    def select_target_clear_ts(x, eps):
        if rng.random() < eps:
            k = int(rng.integers(0, len(AGG_TARGETS)))
            return AGG_TARGETS[k], k
        scores = []
        for k in range(len(AGG_TARGETS)):
            Ainv = np.linalg.inv(agg_bandit.A[k])
            mu   = Ainv @ agg_bandit.b[k]
            theta = rng.multivariate_normal(mu, (agg_bandit.v**2)*Ainv)
            scores.append(theta @ x)
        k = int(np.argmax(scores))
        return AGG_TARGETS[k], k

    def select_incentive_value_aware(ci, x, spread_fore, p_clear_hat, eps):
        if rng.random() < eps:
            a_idx = int(rng.integers(0, len(INCENTIVES)))
            return INCENTIVES[a_idx], a_idx
        best_score = -1e18
        best_a_idx = 0
        for a_idx, a in enumerate(INCENTIVES):
            s_hat = ts_sample_shift(cust_bandits[ci], a_idx, x)
            s_hat = min(s_hat, customers[ci].flex_day)
            unit_margin = spread_fore * p_clear_hat - a
            score = s_hat * unit_margin
            if score > best_score:
                best_score = score
                best_a_idx = a_idx
        return INCENTIVES[best_a_idx], best_a_idx

    def sample_delivered_Q(assignments, x, day_idx_in_paramY, tp):
        Q = 0.0
        for a, lst in assignments.items():
            for (ci, a_idx) in lst:
                m = expected_shift_from_bandit(ci, a_idx, x)
                s = cust_shift_noise[ci][a_idx].std(floor=0.005)
                q = max(0.0, rng.normal(m, s))
                q = min(q, customers[ci].flex_day)
                q = min(q, mtu_cap_kwh(day_idx_in_paramY, tp, ci))
                Q += q
        return Q

    def choose_steps_cvar_joint(q_peak, q_low, Q_total_pred, incentive_avg, assignments, x, day_idx_in_paramY, tp,
                               steps=STEPS_PER_CURVE, alpha=ALPHA_CVAR,
                               spread_z=0.0, target_clear=0.60, MC_SAMPLES=400,
                               penalty_under=PENALTY_UNDER):
        if steps <= 0 or Q_total_pred <= 1e-6:
            return []

        if   spread_z < 0.30:  max_steps = 1
        elif spread_z < 0.80:  max_steps = 2
        elif spread_z < 1.50:  max_steps = 3
        else:                  max_steps = steps

        mu_p, mu_l = q_peak[0.5], q_low[0.5]
        sig_p = max(0.003, (q_peak[0.9]-q_peak[0.1])/(2*1.2816))
        sig_l = max(0.003, (q_low[0.9]-q_low[0.1])/(2*1.2816))

        zgrid = np.random.normal(0, 1, 100000)
        def ppf(p): return float(np.quantile(zgrid, p))

        p_targets = np.array([target_clear+0.15, target_clear+0.05, target_clear-0.05, target_clear-0.15, target_clear-0.25])
        p_targets = np.clip(p_targets, 0.15, 0.90)[:max_steps]
        cand_bids = [mu_p + sig_p * ppf(1.0 - p) for p in p_targets]

        p_peak_s = rng.normal(mu_p, sig_p, size=MC_SAMPLES)
        p_low_s  = rng.normal(mu_l, sig_l, size=MC_SAMPLES)
        spread_s = np.maximum(0.0, p_peak_s - p_low_s)

        Q_deliv_s = np.array([sample_delivered_Q(assignments, x, day_idx_in_paramY, tp) for _ in range(MC_SAMPLES)])

        best_cvar = -1e18
        best_n = 0

        for n in range(1, max_steps+1):
            q_chunk = Q_total_pred / n
            Q_cleared = np.zeros(MC_SAMPLES)
            for i in range(n):
                bid_i = cand_bids[i]
                Q_cleared += q_chunk * (p_peak_s > bid_i)

            Q_exec = np.minimum(Q_deliv_s, Q_cleared)
            under  = np.maximum(0.0, Q_cleared - Q_deliv_s)

            profit_s = (spread_s - incentive_avg) * Q_exec - penalty_under * under
            cvar = cvar_of_profits(profit_s, alpha)

            if cvar > best_cvar:
                best_cvar = cvar
                best_n = n

        if best_n > 0:
            q_chunk = Q_total_pred / best_n
            return [(q_chunk, cand_bids[i]) for i in range(best_n)]

        return []

    # For MTU caps we need mapping day->index in the forecast baseline tensor
    param_day_to_idx = {d: i for i, d in enumerate(param_all_days)}

    # ============================================================
    # 8) RUN SIMULATION (TRAIN then TEST)
    # ============================================================
    def run_sim(days_list, learn=True, past_actual_prices=None, tag="train"):
        if past_actual_prices is None:
            past_actual_prices = []

        day_metrics = []
        pair_log = []
        cust_daily_log = []
        offer_log = []

        for di, day in enumerate(days_list):
            day_df = df[df["date"] == day].copy()
            actual = day_df[price_col].to_numpy(float)

            temp = day_df[temp_col].to_numpy(float) if temp_col in day_df else np.full(96, np.nan)
            if np.isnan(temp).any():
                fill_val = np.nanmean(temp) if np.isfinite(temp).any() else 15.0
                temp = np.where(np.isfinite(temp), temp, fill_val)

            mu, qfore, sigma = build_forecast_quantiles(past_actual_prices, base_sigma=0.006, K=7)

            peak_idx = list(np.argsort(mu)[-N_PEAK:])
            low_idx  = list(np.argsort(mu)[:N_LOW])
            peak_idx.sort()
            low_idx.sort()

            pairs_scored = []
            for tp, tl in zip(reversed(peak_idx), low_idx):
                spread_fore_med = float(qfore[0.5][tp] - qfore[0.5][tl])
                pairs_scored.append((tp, tl, spread_fore_med))

            pairs_scored.sort(key=lambda t: t[2], reverse=True)
            pairs_scored = pairs_scored[:min(MAX_STEPS_PER_DAY, len(pairs_scored))]

            eps_base = eps_schedule(di)

            offers_today = np.zeros(n_cust, dtype=int)
            paid_today = np.zeros(n_cust, dtype=float)
            shifted_today = np.zeros(n_cust, dtype=float)

            daily_profit = 0.0
            daily_revenue = 0.0
            daily_incent_paid = 0.0
            daily_penalty = 0.0

            steps_total_day = 0
            steps_cleared_day = 0

            daily_under = 0.0
            daily_over  = 0.0
            daily_Q_pred = 0.0
            daily_Q_real = 0.0
            daily_Q_exec = 0.0
            daily_Q_cleared = 0.0

            day_pairs = 0
            day_bid_pairs = 0
            day_no_bid_pairs = 0
            day_acc_sum_bid = 0.0
            day_Qpred_sum_bid = 0.0
            day_Qcleared_sum_bid = 0.0

            mean_incent = []
            mean_nonzero_incent = []

            hist_spreads = [mu[tp] - mu[tl] for tp, tl, _ in pairs_scored]
            m_spread = float(np.mean(hist_spreads))
            s_spread = float(np.std(hist_spreads)) + 1e-6

            m_temp = float(np.mean(temp))
            s_temp = float(np.std(temp)) + 1e-6

            day_idx_in_paramY = param_day_to_idx.get(day, None)
            if day_idx_in_paramY is None:
                raise ValueError(f"Day {day} not found in param_all_days; cannot apply MTU cap.")

            for tp, tl, spread_fore_med_ranked in pairs_scored:
                dow = pd.Timestamp(day).weekday()
                hour_peak = (tp*15)/60.0

                spread_mu = mu[tp] - mu[tl]
                z = (spread_mu - m_spread) / s_spread

                hs, hc = cyc_feat(hour_peak, 24.0)
                ds, dc = cyc_feat(dow, 7.0)
                zt = (temp[tp] - m_temp)/s_temp
                x = np.array([1.0, z, z*z, hs, hc, ds, dc, zt], float)

                if learn:
                    eps_target = max(EPS_MIN, eps_base*0.5)
                    eps_incent = max(EPS_MIN, eps_base * (0.5 if z > 1.0 else 1.0))
                else:
                    eps_target = 0.0
                    eps_incent = 0.0

                target_clear, target_idx = select_target_clear_ts(x, eps=eps_target)
                p_clear_hat = float(target_clear)

                spread_fore = max(0.0, float(qfore[0.5][tp] - qfore[0.5][tl]))

                assignments = {}
                chosen_incentives = []

                if spread_fore < LOW_SPREAD_STOP:
                    a0_idx = INCENTIVES.index(0.00)
                    for ci in range(n_cust):
                        assignments.setdefault(0.00, []).append((ci, a0_idx))
                        chosen_incentives.append(0.00)
                else:
                    for ci in range(n_cust):
                        if offers_today[ci] >= MAX_OFFERS_PER_DAY_PER_CUSTOMER:
                            a, a_idx = 0.00, INCENTIVES.index(0.00)
                        else:
                            a, a_idx = select_incentive_value_aware(ci, x, spread_fore, p_clear_hat, eps_incent)

                            if learn and di < WARMUP_DAYS:
                                if rng.random() < WARMUP_P_FORCE_NONZERO and a <= 1e-12:
                                    a_idx = WARMUP_MIN_NONZERO_ARM
                                    a = INCENTIVES[a_idx]
                                if spread_fore * p_clear_hat - a <= 0.0:
                                    a, a_idx = 0.00, INCENTIVES.index(0.00)
                            else:
                                s_mean = expected_shift_from_bandit(ci, a_idx, x)
                                s_mean = min(s_mean, mtu_cap_kwh(day_idx_in_paramY, tp, ci))
                                if s_mean * (spread_fore * p_clear_hat - a) <= 0.0:
                                    a, a_idx = 0.00, INCENTIVES.index(0.00)

                        assignments.setdefault(a, []).append((ci, a_idx))
                        chosen_incentives.append(a)

                        offer_log.append({
                              "tag": tag,
                              "date": str(day),
                              "t_peak": int(tp),
                              "customer": cust_cols[ci],
                              "incentive": float(a)
                          })

                        if a > 1e-12:
                            offers_today[ci] += 1

                mean_incent.append(float(np.mean(chosen_incentives)) if chosen_incentives else 0.0)
                nz = [a for a in chosen_incentives if a > 1e-12]
                mean_nonzero_incent.append(float(np.mean(nz)) if len(nz) else 0.0)

                # Deliverable-aware Q_pred
                Q_total_pred = 0.0
                cost_total_pred = 0.0
                for a, lst in assignments.items():
                    for (ci, a_idx) in lst:
                        s = expected_shift_from_bandit(ci, a_idx, x)
                        s = min(s, mtu_cap_kwh(day_idx_in_paramY, tp, ci))
                        Q_total_pred += s
                        cost_total_pred += a * s
                incentive_avg = (cost_total_pred / Q_total_pred) if Q_total_pred > 1e-9 else 0.0

                q_peak = {q: float(qfore[q][tp]) for q in [0.1,0.2,0.35,0.5,0.65,0.8,0.9]}
                q_low  = {q: float(qfore[q][tl]) for q in [0.1,0.2,0.35,0.5,0.65,0.8,0.9]}

                steps = choose_steps_cvar_joint(
                    q_peak, q_low, Q_total_pred, incentive_avg,
                    assignments, x, day_idx_in_paramY, tp,
                    steps=STEPS_PER_CURVE, alpha=ALPHA_CVAR,
                    spread_z=z, target_clear=target_clear, MC_SAMPLES=MC_SAMPLES,
                    penalty_under=PENALTY_UNDER
                )

                # Always-bid fallback
                if (steps is None) or (len(steps) == 0):
                    if (spread_fore < LOW_SPREAD_STOP) or (Q_total_pred <= 1e-9):
                        steps = []
                    else:
                        q_fallback = float(max(0.0, FALLBACK_Q_FRAC * Q_total_pred))
                        bid_fallback = float(q_peak[FALLBACK_BID_Q])
                        steps = [(q_fallback, bid_fallback)]

                day_pairs += 1
                if (steps is None) or (len(steps) == 0):
                    day_no_bid_pairs += 1
                else:
                    day_bid_pairs += 1

                realized_shifts = np.zeros(n_cust, float)
                inc_cost_total = 0.0

                for a, lst in assignments.items():
                    for (ci, a_idx) in lst:
                        shift_i = customers[ci].realized(a, hour_peak, dow)
                        shift_i = min(shift_i, mtu_cap_kwh(day_idx_in_paramY, tp, ci))

                        realized_shifts[ci] += shift_i
                        shifted_today[ci] += shift_i

                        if learn:
                            cust_bandits[ci].update(a_idx, x, shift_i)
                            pred_mean = expected_shift_from_bandit(ci, a_idx, x)
                            cust_shift_noise[ci][a_idx].update(shift_i - pred_mean)

                        inc_cost_total += a * shift_i

                Q_realized = float(realized_shifts.sum())

                p_peak = float(actual[tp])
                p_low  = float(actual[tl])
                spread_real = max(0.0, p_peak - p_low)

                Q_cleared = 0.0
                n_cleared_steps = 0
                steps_sorted = sorted(steps, key=lambda t: t[1]) if steps else []

                if steps_sorted:
                    for qk, bk in steps_sorted:
                        steps_total_day += 1
                        if p_peak >= bk:
                            steps_cleared_day += 1
                            n_cleared_steps += 1
                            Q_cleared += float(qk)

                acc_ratio = (n_cleared_steps / max(1, len(steps_sorted))) if steps_sorted else 0.0

                Q_exec = min(Q_realized, Q_cleared)
                exec_frac = (Q_exec / Q_realized) if Q_realized > 1e-9 else 0.0

                if steps_sorted:
                    day_acc_sum_bid += acc_ratio
                    day_Qpred_sum_bid += Q_total_pred
                    day_Qcleared_sum_bid += Q_cleared

                revenue = spread_real * Q_exec
                inc_cost_exec = inc_cost_total * exec_frac

                under_delivery = max(0.0, Q_cleared - Q_realized)
                over_delivery  = max(0.0, Q_realized - Q_cleared)
                penalty = PENALTY_UNDER * under_delivery
                profit_pair = revenue - inc_cost_exec - penalty

                if learn:
                    agg_bandit.update(target_idx, x, profit_pair)

                if exec_frac > 0:
                    for a, lst in assignments.items():
                        for (ci, a_idx) in lst:
                            paid_today[ci] += a * (realized_shifts[ci] * exec_frac)

                daily_profit += profit_pair
                daily_revenue += revenue
                daily_incent_paid += inc_cost_exec
                daily_penalty += penalty

                daily_under += under_delivery
                daily_over  += over_delivery
                daily_Q_pred += Q_total_pred
                daily_Q_real += Q_realized
                daily_Q_exec += Q_exec
                daily_Q_cleared += Q_cleared

                # ---- serialize bid steps for logging ----
                steps_txt = ";".join([f"{float(qk):.6f}@{float(bk):.6f}" for (qk, bk) in steps_sorted]) if steps_sorted else ""

                pair_log.append({
                    "tag": tag, "date": str(day),
                    "t_peak": int(tp), "t_low": int(tl),
                    "hour_peak": float(hour_peak), "dow": int(dow),
                    "spread_fore_med": float(spread_fore),
                    "spread_real": float(spread_real),
                    "target_clear": float(target_clear),
                    "incentive_avg_pred": float(incentive_avg),
                    "offers_sent": int(sum(1 for a in chosen_incentives if a > 1e-12)),
                    "pct_zero_incentive": float(np.mean([1.0 if a <= 1e-12 else 0.0 for a in chosen_incentives])),
                    "Q_pred": float(Q_total_pred),
                    "Q_realized": float(Q_realized),
                    "Q_cleared": float(Q_cleared),
                    "Q_exec": float(Q_exec),
                    "exec_frac": float(exec_frac),
                    "under_delivery": float(under_delivery),
                    "over_delivery": float(over_delivery),
                    "n_steps": int(len(steps_sorted)),
                    "n_cleared_steps": int(n_cleared_steps),
                    "acc_ratio_steps": float(acc_ratio),
                    "revenue": float(revenue),
                    "inc_cost_exec": float(inc_cost_exec),
                    "penalty": float(penalty),
                    "profit_pair": float(profit_pair),
                    "profit_per_kWh_exec": float(profit_pair / Q_exec) if Q_exec > 1e-9 else 0.0,
                    "bid_steps": steps_txt
                })

            no_bid_ratio = day_no_bid_pairs / max(1, day_pairs)
            acc_if_bid = day_acc_sum_bid / max(1, day_bid_pairs)
            cleared_over_pred_if_bid = day_Qcleared_sum_bid / max(1e-9, day_Qpred_sum_bid)

            print(
                f"[{tag}] {day} | pairs={day_pairs} | bid_pairs={day_bid_pairs} "
                f"| no-bid%={no_bid_ratio:.2f} | acc|bid={acc_if_bid:.2f} "
                f"| Qcleared/Qpred|bid={cleared_over_pred_if_bid:.2f}"
            )

            day_metrics.append({
                "tag": tag, "date": str(day),
                "profit": float(daily_profit),
                "revenue": float(daily_revenue),
                "incent_paid": float(daily_incent_paid),
                "penalty": float(daily_penalty),
                "steps_total_day": int(steps_total_day),
                "steps_cleared_day": int(steps_cleared_day),
                "accept_ratio_steps": float(steps_cleared_day / max(1, steps_total_day)),
                "mean_incentive": float(np.mean(mean_incent)) if mean_incent else 0.0,
                "mean_nonzero_incentive": float(np.mean(mean_nonzero_incent)) if mean_nonzero_incent else 0.0,
                "eps": float(eps_base if learn else 0.0),
                "Q_pred_sum": float(daily_Q_pred),
                "Q_real_sum": float(daily_Q_real),
                "Q_cleared_sum": float(daily_Q_cleared),
                "Q_exec_sum": float(daily_Q_exec),
                "under_sum": float(daily_under),
                "over_sum": float(daily_over),
                "under_rate": float(daily_under / max(1e-9, daily_Q_cleared)),
                "exec_rate": float(daily_Q_exec / max(1e-9, daily_Q_cleared)),
                "gini_offers": float(gini(offers_today)),
            })

            for ci in range(n_cust):
                cust_daily_log.append({
                    "tag": tag, "date": str(day),
                    "customer": cust_cols[ci],
                    "offers": int(offers_today[ci]),
                    "shift_kWh": float(shifted_today[ci]),
                    "paid_eur": float(paid_today[ci]),
                    "paid_per_kWh": float(paid_today[ci]/shifted_today[ci]) if shifted_today[ci] > 1e-9 else 0.0
                })

            past_actual_prices.append(actual)

            if learn:
                for b in cust_bandits:
                    b.v = max(0.25, b.v * 0.995)
                agg_bandit.v = max(0.25, agg_bandit.v * 0.997)

        return (pd.DataFrame(day_metrics),
                pd.DataFrame(pair_log),
                pd.DataFrame(cust_daily_log),
                pd.DataFrame(offer_log),
                past_actual_prices)


    # ---- Train
    train_metrics, train_pairs, train_cust, train_offers, hist = run_sim(train_days, learn=True, past_actual_prices=[], tag="train")
    # ---- Test
    test_metrics,  test_pairs,  test_cust,  test_offers,  hist = run_sim(test_days, learn=False, past_actual_prices=hist, tag="test")

    offers_df = pd.concat([train_offers, test_offers], ignore_index=True)
    metrics_df = pd.concat([train_metrics, test_metrics], ignore_index=True)
    pairs_df   = pd.concat([train_pairs, test_pairs], ignore_index=True)
    cust_df    = pd.concat([train_cust, test_cust], ignore_index=True)

    print("TRAIN total profit:", train_metrics["profit"].sum())
    print("TEST  total profit:", test_metrics["profit"].sum())
    print("TRAIN avg under_rate:", train_metrics["under_rate"].mean())
    print("TEST  avg under_rate:", test_metrics["under_rate"].mean())

    # -------------------------
    # 9) OUTPUTS
    # -------------------------
    out_dir = "/content" if os.path.exists("/content") else "/mnt/data"

    metrics_path = os.path.join(out_dir, f"{out_prefix}_dr_day_metrics_train_test.csv")
    pairs_path   = os.path.join(out_dir, f"{out_prefix}_dr_pair_log_train_test.csv")
    cust_path    = os.path.join(out_dir, f"{out_prefix}_dr_customer_daily_train_test.csv")

    metrics_df.to_csv(metrics_path, index=False)
    pairs_df.to_csv(pairs_path, index=False)
    cust_df.to_csv(cust_path, index=False)

    print("Saved:",
          metrics_path,
          pairs_path,
          cust_path)

    # -------------------------
    # 9.5) DAY PLAN for target_date
    # -------------------------
    plan_df = build_day_plan(df, pairs_df, target_date_str)

    # ΜΟΝΟ slots που έχουν bids (peak slots)
    bidded = plan_df[(plan_df["bids_steps"].astype(str).str.len() > 0) & (plan_df["bid_Q_total"] > 0)].copy()

    print(f"\n===== BIDDED SLOTS — {out_prefix} — {target_date_str} =====")
    display(bidded[[
        "datetime","slot","t_low","mcp_price","temp_c",
        "bid_Q_total","bids_steps",
        "incentive_avg_pred","offers_sent","pct_zero_incentive","target_clear"
    ]])



    # -------------------------
    # 10) PLOTS (optional)
    # -------------------------
    plt.figure()
    plt.plot(np.arange(len(train_metrics)), train_metrics["profit"].cumsum(), label="Train cum profit")
    plt.plot(len(train_metrics) + np.arange(len(test_metrics)), test_metrics["profit"].cumsum(), label="Test cum profit")
    plt.axvline(x=len(train_metrics)-0.5, linestyle="--")
    plt.title(f"Cumulative Profit (Train vs Test) — {out_prefix}")
    plt.xlabel("Day index (train then test)"); plt.ylabel("EUR"); plt.legend(); plt.show()

    plt.figure()
    plt.plot(metrics_df["under_rate"].to_numpy())
    plt.axvline(x=len(train_metrics)-0.5, linestyle="--")
    plt.title(f"Under-delivery rate (split marked) — {out_prefix}")
    plt.xlabel("Day index (train then test)"); plt.ylabel("Rate"); plt.ylim(0,1); plt.show()

    plt.figure()
    plt.plot(train_metrics["accept_ratio_steps"].rolling(7).mean().to_numpy(), label="Train (7d MA)")
    plt.plot(len(train_metrics) + np.arange(len(test_metrics)),
             test_metrics["accept_ratio_steps"].rolling(7).mean().to_numpy(), label="Test (7d MA)")
    plt.axvline(x=len(train_metrics)-0.5, linestyle="--")
    plt.title(f"Acceptance ratio of bid steps (7d MA) — {out_prefix}")
    plt.xlabel("Day index (train then test)"); plt.ylabel("Acceptance"); plt.ylim(0,1); plt.legend(); plt.show()

    # -------------------------
    # 10.5) DAILY PROFIT PLOT + 30D ROLLING (ORANGE)
    # -------------------------
    # κρατάμε τη σειρά ημερών όπως εμφανίζονται στο metrics_df
    prof = metrics_df["profit"].astype(float).reset_index(drop=True)

    # rolling 30-day mean (min_periods=1 για να φαίνεται από την αρχή)
    roll30 = prof.rolling(30, min_periods=1).mean()

    plt.figure()
    plt.plot(prof.to_numpy(), label="Daily profit")
    plt.plot(roll30.to_numpy(), color="orange", label="Rolling 30d profit (mean)")
    plt.axvline(x=len(train_metrics)-0.5, linestyle="--")
    plt.title(f"Daily Profit + Rolling 30d (split marked) — {out_prefix}")
    plt.xlabel("Day index (train then test)")
    plt.ylabel("EUR/day")
    plt.legend()
    plt.show()


    return {
        "df": df,
        "metrics_df": metrics_df,
        "pairs_df": pairs_df,
        "cust_df": cust_df,
        "plan_df": plan_df,
        "paths": {
            "metrics": metrics_path,
            "pairs": pairs_path,
            "cust": cust_path
        }
    }


In [ ]:
# Example run
CSV_IN = "data_preprocessed.csv"
OUT_PREFIX = "group1"
TARGET_DATE = "2025-09-30"

# Uncomment to run:
# results = run_system(CSV_IN, OUT_PREFIX, target_date_str=TARGET_DATE, SPLIT_YEAR=2025, MAX_DAYS=None, seed=13)
